In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

# Note: dataset must be downloaded first via: python -m src.data.download
DATA_PATH = Path("../data/raw/application_train.csv")
print(f"Dataset exists: {DATA_PATH.exists()}")

In [ ]:
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Shape: {df.shape}")
    print(f"\nTarget distribution:")
    print(df['TARGET'].value_counts(normalize=True).round(4))
else:
    print("\u26a0\ufe0f  Dataset not yet downloaded.")
    print("Run: python -m src.data.download")
    # Create a small synthetic sample for notebook structure demonstration
    import numpy as np
    np.random.seed(42)
    df = pd.DataFrame({
        'SK_ID_CURR': range(100),
        'TARGET': np.random.choice([0, 1], size=100, p=[0.92, 0.08]),
        'AMT_CREDIT': np.random.uniform(100000, 1000000, 100),
        'AMT_INCOME_TOTAL': np.random.uniform(50000, 500000, 100),
        'AMT_ANNUITY': np.random.uniform(10000, 50000, 100),
        'DAYS_BIRTH': np.random.randint(-25000, -8000, 100),
        'DAYS_EMPLOYED': np.random.randint(-5000, 0, 100),
    })
    print(f"Using synthetic sample: {df.shape}")
    print(df['TARGET'].value_counts(normalize=True))

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
print(f"Columns with missing values: {(missing > 0).sum()}")
print("\nTop 20 columns with most missing:")
print(missing.head(20).to_string())

In [ ]:
target_counts = df['TARGET'].value_counts()
print(f"Class 0 (adimplente): {target_counts[0]:,} ({target_counts[0]/len(df):.1%})")
print(f"Class 1 (inadimplente): {target_counts[1]:,} ({target_counts[1]/len(df):.1%})")
print(f"\nDesequil\u00edbrio: {df['TARGET'].mean():.1%} de inadimpl\u00eancia")

In [ ]:
num_cols = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']
available_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(1, len(available_cols), figsize=(18, 4))
if len(available_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, available_cols):
    df[col].hist(bins=30, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Distribui\u00e7\u00f5es das Features Financeiras', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: reports/figures/distributions.png")

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.features import engineer_features

df_eng = engineer_features(df.copy())
new_features = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'AGE_YEARS', 'EMPLOYMENT_YEARS']
available_new = [c for c in new_features if c in df_eng.columns]

print("Engineered features stats:")
print(df_eng[available_new].describe().round(3).to_string())

In [ ]:
numeric_df = df_eng.select_dtypes(include=[np.number])
if 'TARGET' in numeric_df.columns:
    corr_with_target = numeric_df.corr()["TARGET"].drop("TARGET").abs().sort_values(ascending=False)
    print("Top 15 features more correlated with TARGET:")
    print(corr_with_target.head(15).to_string())

In [ ]:
print("=== Business Insights ===")
print()

if 'CREDIT_INCOME_RATIO' in df_eng.columns and 'TARGET' in df_eng.columns:
    ratio_by_target = df_eng.groupby('TARGET')['CREDIT_INCOME_RATIO'].agg(['mean', 'median'])
    print("CREDIT_INCOME_RATIO by TARGET:")
    print(ratio_by_target.round(3).to_string())
    print()

if 'AGE_YEARS' in df_eng.columns and 'TARGET' in df_eng.columns:
    age_by_target = df_eng.groupby('TARGET')['AGE_YEARS'].agg(['mean', 'median'])
    print("AGE_YEARS by TARGET:")
    print(age_by_target.round(2).to_string())

In [ ]:
print("=== EDA Summary ===")
print(f"Dataset shape: {df.shape}")
print(f"Target imbalance: {df['TARGET'].mean():.1%} positive (inadimplente)")
print(f"Features engineered: {len(available_new)}")
print()
print("Key findings:")
print("- High class imbalance (~8%) \u2192 use scale_pos_weight in XGBoost")
print("- CREDIT_INCOME_RATIO and AGE_YEARS show predictive signal")
print("- Many columns have >60% missing \u2192 handled by drop_high_missing()")
print("- DAYS_EMPLOYED sentinel value 365243 (unemployed) \u2192 handled by clip(upper=0)")